<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/quickstart.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · quickstart

A guided tour: assemble one regatta from [scores.collegesailing.org](https://scores.collegesailing.org) into a complete data model, then into a pandas DataFrame.

A regatta's full picture is spread across several pages (an overview, each division, full scores, sailors). The parsers read one page each; the **adapter** stitches them into a single `RegattaScores` model. The parsers do no network I/O — we fetch pages ourselves and hand the HTML in.

## Install

The `[fetch]` extra adds `httpx` — the parsers don't fetch, so we need a client to pull pages ourselves.

In [ ]:
!pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"
# pandas + matplotlib ship with Colab; uncomment elsewhere:
# !pip install -q pandas matplotlib

## Fetch, with a disk cache

Assembling a regatta means several requests, and the site is a little slow, so we wrap fetches in `scraper.cache`: the first call for a URL hits the network, every repeat (later cells, a re-run, a kernel restart) reads from `./.scraper_cache`. The cache **never expires** — ideal for finished regattas; if you're following a *live* one, delete the folder to refresh.

_Colab tip: the cache lives in the ephemeral VM. To keep it across runtime resets, set `SCRAPER_CACHE_DIR` to a mounted-Drive path **before** the import below._

In [ ]:
# import os; from google.colab import drive
# drive.mount('/content/drive')
# os.environ['SCRAPER_CACHE_DIR'] = '/content/drive/MyDrive/icsa_cache'

import httpx

from scraper import cache


def fetch(url: str) -> str:
    html = cache.get(url)
    if html is None:
        html = httpx.get(url, timeout=30).text
        cache.put(url, html)
    return html

## Pick a regatta

Point at any regatta by its season + slug (the URL is `/{season}/{slug}/`). Here, the 2026 Open Fleet Race National Championships. `scraper.parsers.season.parse` lists every regatta in a season if you want to browse.

In [ ]:
BASE = "https://scores.collegesailing.org"
SEASON = "s26"  # f25 = Fall 2025, s26 = Spring 2026
SLUG = "open-fleet-race-national-championships"

## The overview page is an index

`regatta.parse` reads the overview: name, scoring type, hosts, dates, and — crucially — *which sub-pages exist*. That tells you what to fetch next.

In [ ]:
from scraper.parsers import regatta

meta = regatta.parse(fetch(f"{BASE}/{SEASON}/{SLUG}/"), SEASON, SLUG)

print("Name:        ", meta.name)
print("Scoring:     ", meta.scoring, "/", meta.participant)
print("Hosts:       ", ", ".join(meta.hosts))
print("Dates:       ", meta.start_time[:10], "->", meta.end_date)
print("Divisions:   ", meta.division_pages)
print("Full scores: ", meta.has_full_scores_page)
print("Sailors:     ", meta.has_sailors_page)
print("Team racing: ", meta.has_all_races_page)

## Assemble the full picture

For fleet racing, the `/full-scores/` page carries every division's per-race results together. `full_scores.parse` extracts the raw rows; the adapter's `build_fleet_scores` stitches them into a `RegattaScores` — one `TeamResult` per team, each with its divisions and every race.

_(Team-racing regattas — where `meta.has_all_races_page` is true — use the `/all/` page with `adapter.build_team_scores` instead.)_

In [ ]:
from scraper.adapter import build_fleet_scores
from scraper.parsers import full_scores

fs_html = fetch(f"{BASE}/{SEASON}/{SLUG}/full-scores/")
div_scores = full_scores.parse(fs_html)
reg = build_fleet_scores(fs_html, SEASON, SLUG, div_scores)

print("Regatta:     ", reg.name)
print("Scoring:     ", reg.scoring_type)
print("Final:       ", reg.is_final)
print("Races sailed:", reg.races_sailed)
print("Teams:       ", len(reg.teams))

## Standings

Teams come back already placed. Each `TeamResult` carries its per-division totals — note the overall winner needn't lead any single division.

In [ ]:
divs = sorted({d for t in reg.teams for d in t.divisions})
head = "  ".join(f"{d:>4}" for d in divs)
print(f"{'#':>3}  {'School':<22} {head}  {'Total':>6}")
for t in reg.teams:
    cells = "  ".join(
        f"{t.divisions[d].total:>4}" if d in t.divisions else f"{'-':>4}" for d in divs
    )
    print(f"{t.place:>3}  {t.school:<22} {cells}  {t.total:>6}")

## Every race → a tidy DataFrame

Walk the model down to `RaceScore` for one row per (school, division, race) — the long format pandas likes, penalties included.

In [ ]:
import pandas as pd

records = []
for t in reg.teams:
    for dname, dres in t.divisions.items():
        for r in dres.races:
            records.append(
                {
                    "school": t.school,
                    "slug": t.school_slug,
                    "division": dname,
                    "race": r.race_num,
                    "points": r.points,
                    "penalty": r.penalty,
                }
            )

df = pd.DataFrame.from_records(records)
print(df.shape)
df.head(10)

## Combined standings

Pivot to one row per school with each division's contribution, then sum — lowest total wins. (Tied totals may order differently from `reg.teams`, which applies the official sailing tiebreakers.)

In [ ]:
standings = df.pivot_table(index="school", columns="division", values="points", aggfunc="sum")
standings["Total"] = standings.sum(axis=1)
standings = standings.sort_values("Total")
standings.insert(0, "Rank", range(1, len(standings) + 1))
standings

## Quick chart

In [ ]:
import matplotlib.pyplot as plt

top = standings.head(10).iloc[::-1]
plt.figure(figsize=(8, 4))
plt.barh(top.index, top["Total"])
plt.xlabel("Total points (lower = better)")
plt.title(f"{reg.name} - top 10")
plt.tight_layout()
plt.show()

---
**Recap of the assembly pattern:** overview (`regatta.parse`) tells you the scoring type and which pages exist → fetch those pages → a parser reads each → the adapter (`build_fleet_scores` / `build_team_scores`) stitches them into one `RegattaScores`. Swap `SLUG`/`SEASON` to explore any regatta.